## Fazer um Model Register e registar uma pipepline no mlflow

In [1]:
import mlflow
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [2]:
ROOT_PATH = '../../data/'
SEED = 42
df = pd.read_csv(root_path + 'lending_data.csv')
TARGET_COL = "default.payment.next.month"

NameError: name 'root_path' is not defined

## Definir a diretoria onde as experiências são guardadas

In [ ]:
from pathlib import Path

uri = "../../mlruns"

Path(uri).mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(uri)

### Fazer set da experiência "Rumos Bank Experiment"

In [ ]:
mlflow.set_experiment("Rumos Bank Experiment")

## Criar os datasets

train_set, test_set = train_test_split(df, test_size = 0.2, random_state = seed)

X_train = train_set.drop(['default.payment.next.month'], axis = 'columns')
y_train = train_set['default.payment.next.month']

X_test = test_set.drop(['default.payment.next.month'], axis = 1)
y_test = test_set['default.payment.next.month']

## Criar uma run

In [ ]:
run = mlflow.start_run(run_name="Linear Regression Run - C0.1 - pipeline")
RUN_ID = run.info.run_uuid
RUN_ID

## Guardar datasets, modelos, artefactos, métricas e parametros da run

In [ ]:
# guardarmos o dataset de treino e de teste associado à run
train_dataset = mlflow.data.from_pandas(train_set, targets='default.payment.next.month', name="lending_data")
test_dataset = mlflow.data.from_pandas(test_set, targets='default.payment.next.month', name="lending_data")
mlflow.log_input(train_dataset, context="train")
mlflow.log_input(test_dataset, context="test")

# Guardamos a seed utilizado como parametro
mlflow.log_param("seed", SEED)

## Pipeline e registro do melhor modelo

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("random_forest", RandomForestClassifier(random_state=SEED, n_estimators=100))
])
rf_pipeline.fit(X_train, y_train)
mlflow.sklearn.log_model(rf_pipeline, artifact_path="rf_pipeline", registered_model_name="random_forest")
rf_pipeline

In [ ]:
params=rf_pipeline.get_params()

modified_params = {}
for k, v in params.items():
    new_key = k.replace("random_forest__", '')
    modified_params[new_key] = v

mlflow.log_params(modified_params)
modified_params

In [ ]:
y_preds = rf_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_preds)
mlflow.log_metric("accuracy", acc)
acc

## Terminar a run

In [ ]:
mlflow.end_run()